# 🕵️ Sarcasm Detection with Context-Aware Transformers

Welcome to the end-to-end sarcasm detection project! This notebook contains the entire machine learning pipeline, from data ingestion to model explainability. 

### Objective:
The goal is to detect sarcasm in news headlines by analyzing the incongruity between the **headline** and its **article context**. We compare two architectures:
1. **Baseline Model**: Uses DistilBERT on headlines only.
2. **Fusion Model**: A dual-encoder architecture that fuses headline and article context features.

---

## 1. Environment & Dependencies
First, we mount Google Drive to save our trained models and results permanently, and install the necessary libraries.

In [ ]:
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/sarcasm_detection_project'
os.makedirs(PROJECT_PATH, exist_ok=True)
%cd $PROJECT_PATH

# 2. Install Dependencies
!pip install transformers newspaper3k shap beautifulsoup4 tqdm pandas scikit-learn matplotlib seaborn

# 3. Global Imports
import torch
import torch.nn as nn
import json
import requests
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from transformers import DistilBertModel, DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

## 2. Data Loading & Scraping
Our dataset consists of headlines. To improve detection, we scrape or generate context for each headline.

In [ ]:
from newspaper import Article

class SarcasmDataLoader:
    def __init__(self, file_path, subset_size=None):
        self.file_path = file_path
        self.subset_size = subset_size
        self.df = None

    def load_data(self):
        print(f"Loading data from {self.file_path}...")
        data = []
        with open(self.file_path, 'r') as f:
            for line in f: data.append(json.loads(line))
        self.df = pd.DataFrame(data)
        if self.subset_size: self.df = self.df.sample(n=self.subset_size, random_state=42).reset_index(drop=True)
        return self.df

    def add_context(self, scrape_limit=100):
        print(f"Adding context (Scrape limit: {scrape_limit})...")
        contexts = []
        for i, row in tqdm(self.df.iterrows(), total=len(self.df)):
            headline, url = row['headline'], row['article_link']
            if i < scrape_limit:
                try:
                    article = Article(url)
                    article.download(); article.parse()
                    context = article.text[:300].replace('\n', ' ')
                    if len(context) < 10: context = self._generate_synthetic_context(headline, url)
                except: context = self._generate_synthetic_context(headline, url)
            else: context = self._generate_synthetic_context(headline, url)
            contexts.append(context)
        self.df['context'] = contexts
        return self.df

    def _generate_synthetic_context(self, headline, url):
        domain = "Satire" if "theonion.com" in url else "News"
        return f"Source: {domain}. Topic analysis of: {headline}."

# --- Data Ingestion --- 
DATA_FILE = 'Sarcasm_Headlines_Dataset.json'
if not os.path.exists(DATA_FILE):
    from google.colab import files
    print("Please upload Sarcasm_Headlines_Dataset.json")
    files.upload()

## 3. Preprocessing & Dataset Construction
We use the DistilBERT tokenizer to convert text into token IDs and attention masks.

In [ ]:
class SarcasmDataset(Dataset):
    def __init__(self, headlines, contexts, labels, tokenizer, hl_max_len=64, ctx_max_len=128):
        self.headlines, self.contexts, self.labels = headlines, contexts, labels
        self.tokenizer, self.hl_max_len, self.ctx_max_len = tokenizer, hl_max_len, ctx_max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, item):
        hl_enc = self.tokenizer(str(self.headlines[item]), max_length=self.hl_max_len, padding="max_length", truncation=True, return_tensors='pt')
        ctx_enc = self.tokenizer(str(self.contexts[item]), max_length=self.ctx_max_len, padding="max_length", truncation=True, return_tensors='pt')
        return {
            'hl_input_ids': hl_enc['input_ids'].flatten(),
            'hl_attention_mask': hl_enc['attention_mask'].flatten(),
            'ctx_input_ids': ctx_enc['input_ids'].flatten(),
            'ctx_attention_mask': ctx_enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.float)
        }

def prepare_loaders(df, tokenizer, batch_size=16):
    df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_sarcastic'])
    df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp['is_sarcastic'])
    
    def create_dl(data_df):
        ds = SarcasmDataset(data_df.headline.to_numpy(), data_df.context.to_numpy(), data_df.is_sarcastic.to_numpy(), tokenizer)
        return DataLoader(ds, batch_size=batch_size)
    
    return create_dl(df_train), create_dl(df_val), create_dl(df_test)

## 4. Model Architectures
We define our two models using pre-trained DistilBERT layers.

In [ ]:
class SarcasmBaselineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(768, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        x = self.dropout(outputs[0][:, 0, :])
        return self.sigmoid(self.out(x))

class SarcasmFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.headline_encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.context_encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.fc1 = nn.Linear(768 * 3, 256)
        self.relu = nn.ReLU(); self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(256, 1); self.sigmoid = nn.Sigmoid()

    def forward(self, hl_input_ids, hl_attention_mask, ctx_input_ids, ctx_attention_mask):
        h1 = self.headline_encoder(hl_input_ids, hl_attention_mask)[0][:, 0, :]
        h2 = self.context_encoder(ctx_input_ids, ctx_attention_mask)[0][:, 0, :]
        fusion = torch.cat([h1, h2, torch.abs(h1-h2)], dim=1)
        x = self.dropout(self.relu(self.fc1(fusion)))
        return self.sigmoid(self.out(x))

## 5. Training Logic
Standard PyTorch training loops with validation and model saving.

In [ ]:
def train_epoch(model, loader, optimizer, device, is_fusion=False):
    model.train(); losses = []; correct = 0
    criterion = nn.BCELoss()
    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()
        labels = batch['labels'].to(device).unsqueeze(1)
        if is_fusion:
            outputs = model(batch['hl_input_ids'].to(device), batch['hl_attention_mask'].to(device),
                            batch['ctx_input_ids'].to(device), batch['ctx_attention_mask'].to(device))
        else:
            outputs = model(batch['hl_input_ids'].to(device), batch['hl_attention_mask'].to(device))
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
        correct += torch.sum((outputs > 0.5).float() == labels)
    return correct.double() / len(loader.dataset), np.mean(losses)

def run_training(model, train_loader, val_loader, optimizer, device, epochs=3, name="model", is_fusion=False):
    best_acc = 0
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        t_acc, t_loss = train_epoch(model, train_loader, optimizer, device, is_fusion)
        # Simple evaluation logic embedded for brevity
        model.eval(); v_correct = 0
        with torch.no_grad():
            for b in val_loader:
                lbls = b['labels'].to(device).unsqueeze(1)
                if is_fusion: outs = model(b['hl_input_ids'].to(device), b['hl_attention_mask'].to(device), b['ctx_input_ids'].to(device), b['ctx_attention_mask'].to(device))
                else: outs = model(b['hl_input_ids'].to(device), b['hl_attention_mask'].to(device))
                v_correct += torch.sum((outs > 0.5).float() == lbls)
        v_acc = v_correct.double() / len(val_loader.dataset)
        print(f"Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f}")
        if v_acc > best_acc:
            torch.save(model.state_dict(), f'best_{name}.bin')
            best_acc = v_acc
    return best_acc

## 6. Evaluation Utilities
Functions to generate performance metrics and visual reports.

In [ ]:
def get_metrics_report(model, loader, device, is_fusion=False, name="Model"):
    model.eval(); preds, actuals = [], []
    with torch.no_grad():
        for b in loader:
            if is_fusion: o = model(b['hl_input_ids'].to(device), b['hl_attention_mask'].to(device), b['ctx_input_ids'].to(device), b['ctx_attention_mask'].to(device))
            else: o = model(b['hl_input_ids'].to(device), b['hl_attention_mask'].to(device))
            preds.extend((o > 0.5).float().cpu().numpy())
            actuals.extend(b['labels'].numpy())
    
    y_true, y_pred = np.array(actuals), np.array(preds).flatten()
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}'); plt.show()
    
    return {"Accuracy": accuracy_score(y_true, y_pred), "F1": f1_score(y_true, y_pred)}

## 7. Model Explainability (SHAP)
Using SHAP to understand which tokens influence the model's sarcasm prediction.

In [ ]:
import shap

def explain_prediction(model, tokenizer, text, device):
    model.eval()
    def predict(texts):
        tokens = tokenizer(texts.tolist() if isinstance(texts, np.ndarray) else texts, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
        with torch.no_grad():
            return model(tokens['input_ids'], tokens['attention_mask']).cpu().numpy()
    
    explainer = shap.Explainer(predict, tokenizer)
    shap_values = explainer([text])
    try:
        shap.initjs()
        shap.plots.text(shap_values)
    except: # Fallback for non-interactive environments
        print("SHAP visualization generated. (Interactive plot requires JS environment)")

## 8. Main Execution Pipeline
Running the entire workflow from scratch.

In [ ]:
# 1. Configuration
BATCH_SIZE = 32
EPOCHS = 3
SUBSET_SIZE = 1000 # Using subset for demonstration speed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load Data
loader = SarcasmDataLoader(DATA_FILE, subset_size=SUBSET_SIZE)
df = loader.load_data()
df = loader.add_context(scrape_limit=50)

# 3. Prepare Loaders
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
train_loader, val_loader, test_loader = prepare_loaders(df, tokenizer, batch_size=BATCH_SIZE)

# 4. Train Baseline
print("\n--- Training Baseline Model ---")
baseline_model = SarcasmBaselineModel().to(DEVICE)
opt_b = torch.optim.AdamW(baseline_model.parameters(), lr=2e-5)
run_training(baseline_model, train_loader, val_loader, opt_b, DEVICE, epochs=EPOCHS, name="baseline")

# 5. Train Fusion
print("\n--- Training Fusion Model ---")
fusion_model = SarcasmFusionModel().to(DEVICE)
opt_f = torch.optim.AdamW(fusion_model.parameters(), lr=2e-5)
run_training(fusion_model, train_loader, val_loader, opt_f, DEVICE, epochs=EPOCHS, name="fusion", is_fusion=True)

# 6. Evaluation
m_b = get_metrics_report(baseline_model, test_loader, DEVICE, name="Baseline")
m_f = get_metrics_report(fusion_model, test_loader, DEVICE, is_fusion=True, name="Fusion")

print("\nBenchmark Results:")
print(pd.DataFrame([m_b, m_f], index=['Baseline', 'Fusion']))

# 7. Explainability
explain_prediction(baseline_model, tokenizer, "This is totally not a sarcastic headline.", DEVICE)